In [ ]:
import os
import numpy as np
import pandas as pd
import mysql.connector
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Chargement des variables d'environnement depuis le fichier .env du dossier parent
load_dotenv(dotenv_path="../.env")

In [ ]:
# Connexion à la base de données Cloud Aiven MySQL
connection = mysql.connector.connect(
    host=os.getenv("AIVEN_HOST"),
    port=int(os.getenv("AIVEN_PORT", 14198)),
    user=os.getenv("AIVEN_USER"),
    password=os.getenv("AIVEN_PASSWORD"),
    database="defaultdb"
)

In [ ]:
# 1. Extraction d'un échantillon propre
# Note: On utilise la station 3012 (Saxe-Gambetta) ou 1034 qui contiennent des milliers d'enregistrements.
query = """
SELECT date_enregistrement AS date_enregistrement, velos 
FROM stations_historique 
WHERE id = 3012 
ORDER BY date_enregistrement ASC
"""

In [ ]:
# 2. Chargement dans un DataFrame
df = pd.read_sql(query, connection)
connection.close()

In [ ]:
# Vérification rapide du chargement
print("Colonnes chargées :", list(df.columns))
print("Nombre de lignes brutes :", len(df))
if len(df) > 0:
    print(df.head())

In [ ]:
# 3. Préparation des données (Index temporel + rééchantillonnage dynamique)
if 'date_enregistrement' in df.columns:
    df['date_enregistrement'] = pd.to_datetime(df['date_enregistrement'])
    df.set_index('date_enregistrement', inplace=True)

# Calcul de la durée totale des données chargées en heures
time_span_hours = (df.index.max() - df.index.min()).total_seconds() / 3600

# Choix dynamique de la fréquence de rééchantillonnage pour conserver assez de points (min 48)
if time_span_hours < 8:
    freq = '1T'  # Toutes les minutes
    print(f"Durée des données courte ({time_span_hours:.1f}h). Rééchantillonnage par minute ('1T')")
elif time_span_hours < 36:
    freq = '10T' # Toutes les 10 minutes
    print(f"Durée des données moyenne ({time_span_hours:.1f}h). Rééchantillonnage par 10 minutes ('10T')")
elif time_span_hours < 120:
    freq = '30T' # Toutes les 30 minutes
    print(f"Durée des données ({time_span_hours:.1f}h). Rééchantillonnage par 30 minutes ('30T')")
else:
    freq = 'h'   # Toutes les heures
    print(f"Durée des données longue ({time_span_hours:.1f}h). Rééchantillonnage par heure ('h')")

df = df.resample(freq).mean().ffill()
print(f"Nombre de points rééchantillonnés : {len(df)}")

In [ ]:
# 4. Ingénierie des caractéristiques (Feature Engineering)
if len(df) >= 10:
    # Création des décalages temporels (Lags)
    df['velos_lag1'] = df['velos'].shift(1)
    df['velos_lag2'] = df['velos'].shift(2)
    df['velos_lag3'] = df['velos'].shift(3)
    
    # Extraction des variables de contexte temporel
    df['heure'] = df.index.hour
    df['jour_semaine'] = df.index.dayofweek
    
    # Suppression des lignes contenant des NaN (créés par les décalages)
    df_ml = df.dropna()
    
    print(f"Taille du dataset après ingénierie des caractéristiques : {len(df_ml)} lignes.")
    print("Colonnes explicatives créées : ['velos_lag1', 'velos_lag2', 'velos_lag3', 'heure', 'jour_semaine']")
else:
    print("Pas assez de données pour l'ingénierie des caractéristiques.")

In [ ]:
# 5. Division Entraînement / Test et Entraînement de la Forêt Aléatoire (Random Forest)
if 'df_ml' in locals() and len(df_ml) >= 10:
    # Définition des entrées (X) et de la cible à prédire (y)
    features = ['velos_lag1', 'velos_lag2', 'velos_lag3', 'heure', 'jour_semaine']
    X = df_ml[features]
    y = df_ml['velos']
    
    # Split temporel (80% entraînement, 20% test) pour respecter la nature temporelle des données
    train_size = int(len(df_ml) * 0.8)
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
    
    print(f"Entraînement sur {len(X_train)} points, Test sur {len(X_test)} points.")
    
    # Initialisation et ajustement du modèle Random Forest
    model_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    model_rf.fit(X_train, y_train)
    
    # Prédictions sur le jeu de test
    predictions = model_rf.predict(X_test)
    
    # Calcul des erreurs
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    
    print(f"\n--- Métriques de validation (Random Forest) ---")
    print(f"Erreur Moyenne Absolue (MAE) : {mae:.2f} vélos")
    print(f"Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} vélos")
    
    # Graphique de comparaison
    plt.figure(figsize=(12, 6))
    plt.plot(y_train.index, y_train, label='Entraînement', color='blue')
    plt.plot(y_test.index, y_test, label='Valeurs Réelles', color='green')
    plt.plot(y_test.index, predictions, label='Prédictions Random Forest', color='red', linestyle='--')
    plt.title("Modèle Machine Learning : Random Forest Regressor")
    plt.xlabel("Date / Heure")
    plt.ylabel("Nombre de vélos")
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Pas assez de données pour entraîner le modèle Random Forest.")

In [ ]:
# 6. Importance des caractéristiques (Feature Importance)
if 'model_rf' in locals():
    # Analyse des variables qui ont le plus d'impact sur la décision de l'algorithme
    importances = model_rf.feature_importances_
    indices = np.argsort(importances)
    
    plt.figure(figsize=(10, 5))
    plt.title("Importance des caractéristiques - Random Forest")
    plt.barh(range(len(indices)), importances[indices], color='teal', align='center')
    plt.yticks(range(len(indices)), [features[i] for i in indices])
    plt.xlabel('Importance relative')
    plt.grid(axis='x', linestyle='--')
    plt.show()
    
    for i in reversed(indices):
        print(f"{features[i]} : {importances[i]*100:.1f}%")